# ML-10 — Content Action Playbook

This notebook turns the validated Week-5/6 Logistic Regression output into a **human-reviewed, non-production** review queue. It uses the starter snapshot and a current-state decline proxy. The model score is one directional prioritization input; it is not a future forecast, a causal result, or authorization to change content automatically.

## 1. Ranked actions + reason codes

I refit the already-audited feature pipeline on the full starter snapshot only to create a practice queue after validation was completed in Week 6. The queue combines model probability with a visibility proxy, then caps each pseudonymized client at three rows so one client cannot fill the entire review list. It deliberately omits the label, `trend_direction`, `trend_pct`, all recent comparison-window fields, and IDs from the displayed notebook table.

**Reason-code rules:** `HIGH_MODEL_RISK` means the score is in the candidate pool; `HIGH_VISIBILITY` means 90-day impressions are in the candidate pool's upper quartile; `STALE_CONTENT` means 180+ days since update; `POSITION_OPPORTUNITY` means average position is 8–20; `LOW_ENGAGEMENT` means engagement is below the candidate median. These are descriptive triage cues, not diagnoses or causal explanations.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists())
output_dir = repo_root / 'work' / 'outputs'
figures_dir = repo_root / 'work' / 'figures'
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(repo_root / 'data/raw/content_refresh_anonymized.csv')

feature_cols = [
    'search_volume', 'competition', 'cpc', 'word_count', 'impressions_90d',
    'clicks_90d', 'sessions_90d', 'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
target_col = 'decline_proxy_rule'
df[target_col] = df['trend_direction'].eq('down').astype(int)

preprocess = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
        ('scaler', StandardScaler()),
    ]), feature_cols),
])
model = Pipeline([
    ('preprocess', preprocess),
    ('logistic', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_SEED)),
])
model.fit(df[feature_cols], df[target_col])
df['review_score'] = model.predict_proba(df[feature_cols])[:, 1]
print(f'Scored {len(df):,} starter-snapshot pages across {df.client_id.nunique()} pseudonymized clients.')
print('This refit creates a practice review queue; Week 6 contains the held-out grouped-client evaluation.')

Scored 30,000 starter-snapshot pages across 32 pseudonymized clients.
This refit creates a practice review queue; Week 6 contains the held-out grouped-client evaluation.


In [2]:
# Practical candidate screen: enough observed visibility for a reviewer to assess.
candidates = df[df['impressions_90d'] >= 500].copy()
visibility_q75 = candidates['impressions_90d'].quantile(0.75)
engagement_median = candidates['engagement_rate'].median()

# Opportunity score is a transparent ordering rule, not a predicted outcome or value estimate.
candidates['priority_score'] = (
    candidates['review_score']
    * (1 + np.log1p(candidates['impressions_90d']) / np.log1p(candidates['impressions_90d']).max())
)
candidates['is_high_visibility'] = candidates['impressions_90d'] >= visibility_q75
candidates['is_stale'] = candidates['days_since_last_update'] >= 180
candidates['is_position_opportunity'] = candidates['avg_position'].between(8, 20, inclusive='both')
candidates['is_low_engagement'] = candidates['engagement_rate'].fillna(engagement_median) < engagement_median

def assign_archetype(row):
    if row.is_stale and row.is_high_visibility:
        return 'stale, high-visibility'
    if row.is_position_opportunity and row.is_high_visibility:
        return 'high-visibility position opportunity'
    if row.is_low_engagement:
        return 'low-engagement review candidate'
    return 'model-flagged review candidate'

def assign_action(row):
    if row['archetype'] == 'stale, high-visibility':
        return 'Review refresh brief and current intent'
    if row['archetype'] == 'high-visibility position opportunity':
        return 'Review SERP, intent, and on-page gaps'
    if row['archetype'] == 'low-engagement review candidate':
        return 'Review usefulness and engagement context'
    return 'Diagnose before proposing a content change'

def reason_codes(row):
    codes = ['HIGH_MODEL_RISK']
    if row.is_high_visibility:
        codes.append('HIGH_VISIBILITY')
    if row.is_stale:
        codes.append('STALE_CONTENT')
    if row.is_position_opportunity:
        codes.append('POSITION_OPPORTUNITY')
    if row.is_low_engagement:
        codes.append('LOW_ENGAGEMENT')
    return ' | '.join(codes)

candidates['archetype'] = candidates.apply(assign_archetype, axis=1)
candidates['recommended_human_action'] = candidates.apply(assign_action, axis=1)
candidates['reason_codes'] = candidates.apply(reason_codes, axis=1)
candidates['estimated_review_hours'] = candidates['archetype'].map({
    'stale, high-visibility': 2.0,
    'high-visibility position opportunity': 1.5,
    'low-engagement review candidate': 1.5,
    'model-flagged review candidate': 1.0,
}).astype(float)

# Diversity cap is applied after scoring; identifiers remain in the export only so a reviewer can locate a page.
queue = (candidates.sort_values('priority_score', ascending=False)
        .groupby('client_id', group_keys=False).head(3)
        .sort_values('priority_score', ascending=False).head(50).copy())
queue.insert(0, 'queue_rank', range(1, len(queue) + 1))
queue['value_proxy'] = '90-day impressions; not revenue or expected lift'
queue['review_rule'] = 'Human must verify evidence before any action'

safe_queue_view = queue[[
    'queue_rank', 'archetype', 'recommended_human_action', 'reason_codes',
    'review_score', 'impressions_90d', 'avg_position', 'days_since_last_update',
    'estimated_review_hours', 'value_proxy'
]].head(10).copy()
safe_queue_view['review_score'] = safe_queue_view['review_score'].round(3)
display(safe_queue_view)
print(f'Queue size: {len(queue)} pages; estimated first-pass human review time: {queue.estimated_review_hours.sum():.1f} hours.')

,queue_rank,archetype,recommended_human_action,reason_codes,review_score,impressions_90d,avg_position,days_since_last_update,estimated_review_hours,value_proxy
29400,1,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.846,443434,27.9,104,1.0,90-day impressions; not revenue or expected lift
27178,2,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.807,140079,7.6,20,1.0,90-day impressions; not revenue or expected lift
7445,3,high-visibility position opportunity,"Review SERP, intent, and on-page gaps",HIGH_MODEL_RISK | HIGH_VISIBILITY | POSITION_O...,0.755,208678,9.7,104,1.5,90-day impressions; not revenue or expected lift
3331,4,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.764,128068,2.2,104,1.0,90-day impressions; not revenue or expected lift
482,5,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.753,112434,7.2,20,1.0,90-day impressions; not revenue or expected lift
6903,6,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.725,223271,7.8,20,1.0,90-day impressions; not revenue or expected lift
8954,7,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.811,14830,2.4,104,1.0,90-day impressions; not revenue or expected lift
18402,8,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.785,19572,7.0,104,1.0,90-day impressions; not revenue or expected lift
22631,9,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.775,22080,21.6,20,1.0,90-day impressions; not revenue or expected lift
29978,10,model-flagged review candidate,Diagnose before proposing a content change,HIGH_MODEL_RISK | HIGH_VISIBILITY,0.769,22425,7.6,20,1.0,90-day impressions; not revenue or expected lift


Queue size: 50 pages; estimated first-pass human review time: 53.5 hours.


### Decay / refresh insight

The playbook treats **staleness as a reason to inspect**, not as proof that a refresh will recover visibility. Pages that are both stale and highly visible are placed in a refresh-brief review archetype because they combine observed exposure with time since the last update. Before changing anything, the reviewer must check current search intent, page quality, technical/indexing issues, seasonality, and whether a change is warranted. No observed snapshot association establishes that refreshing causes recovery.

## 2. Intended use and limits

**Intended use:** A content strategist uses the queue weekly or monthly to choose a small, diverse set of pages for manual investigation. The reason codes help the strategist form a review hypothesis; they are not instructions to publish edits. The visibility proxy supports triage toward pages where a review may be more consequential, without estimating revenue or uplift.

**Limits:** The model was evaluated on one held-out set of pseudonymized clients in a starter snapshot, using a current-state proxy derived from trend direction. It was not tested on future periods, new real-world clients, or post-refresh outcomes. Its grouped ROC AUC was 0.589 and average precision 0.600 against a 0.517 test base rate in Week 6. The score therefore offers modest, directional ranking signal only.

In [3]:
use_limits = pd.DataFrame([
    ('Use', 'Prioritize a human review queue; compare reasons with page and SERP evidence.'),
    ('Do not infer', 'A future decline, a causal driver, revenue, or expected lift from this score.'),
    ('Scope', 'Starter snapshot and pseudonymized-client validation only.'),
    ('Decision threshold', 'No automatic threshold; a person decides whether any proposed action is justified.'),
    ('Cost/value', 'Use 90-day impressions as an exposure proxy alongside estimated review hours; do not treat it as money.'),
], columns=['topic', 'playbook rule'])
display(use_limits)

,topic,playbook rule
0,Use,Prioritize a human review queue; compare reaso...
1,Do not infer,"A future decline, a causal driver, revenue, or..."
2,Scope,Starter snapshot and pseudonymized-client vali...
3,Decision threshold,No automatic threshold; a person decides wheth...
4,Cost/value,Use 90-day impressions as an exposure proxy al...


## 3. Human review + the no-go list

Every queued row needs a reviewer to examine the live page and current search context before any action. The reviewer should confirm: (1) the page is still indexed and technically healthy; (2) the decline/reason is not a reporting anomaly, seasonality, or site-wide event; (3) current SERP intent and competitors justify a change; (4) the proposed edit is factually accurate, useful, and brand-appropriate; and (5) the page does not create legal, medical, financial, compliance, or reputation risk. Record the reviewer decision and rationale outside this practice queue.

**No-go: do not automate** publishing, deleting, redirecting, changing claims or prices, modifying regulated/high-risk content, deciding a refresh caused an outcome, or contacting a client. Do not act on rows with insufficient data, sudden tracking changes, a probable technical incident, or unclear intent. Escalate those cases to the responsible human team.

In [4]:
review_checklist = pd.DataFrame([
    ('Evidence check', 'Confirm analytics coverage, indexing, and absence of a measurement break.'),
    ('Search context', 'Inspect current SERP, intent, seasonality, and competing pages.'),
    ('Content decision', 'Choose refresh, no change, technical investigation, or defer; record rationale.'),
    ('Safety escalation', 'Route regulated, legal, financial, medical, brand-sensitive, or client-impacting changes to an owner.'),
    ('Never automate', 'Publishing, deletion, redirects, claim changes, client contact, or causal attribution.'),
], columns=['review step', 'required human action'])
display(review_checklist)

,review step,required human action
0,Evidence check,"Confirm analytics coverage, indexing, and abse..."
1,Search context,"Inspect current SERP, intent, seasonality, and..."
2,Content decision,"Choose refresh, no change, technical investiga..."
3,Safety escalation,"Route regulated, legal, financial, medical, br..."
4,Never automate,"Publishing, deletion, redirects, claim changes..."


## 4. Monitoring / retrain triggers

Monitor the queue as a decision-support tool, not as an unattended production system. On each review cycle, compare the score distribution, reason-code mix, candidate volume, and reviewer outcomes with the prior cycle. Re-evaluate the model only when a later, properly time-aligned outcome is available. A retrain must repeat leakage checks and grouped or time-aware validation before a new queue is used.

In [5]:
monitoring_plan = pd.DataFrame([
    ('Each cycle', 'Queue size, score distribution, reason-code mix, and percentage deferred by reviewers.', 'Pause automated scheduling; investigate data or portfolio change.'),
    ('Each cycle', 'Missingness, zero-position rate, and sudden metric coverage changes.', 'Stop scoring affected rows and investigate instrumentation.'),
    ('When later labels exist', 'Grouped/time-aware precision@K, ROC AUC, average precision, and base rate.', 'Retrain only if validation and leakage audit are repeated.'),
    ('On major change', 'Search, tracking, site, content-template, or client-mix changes.', 'Treat prior estimates as stale; revalidate before reuse.'),
    ('After actions', 'Reviewer decisions and outcomes, with selection bias documented.', 'Do not claim impact without an appropriate experiment or comparison design.'),
], columns=['cadence', 'monitor', 'trigger / response'])
display(monitoring_plan)

,cadence,monitor,trigger / response
0,Each cycle,"Queue size, score distribution, reason-code mi...",Pause automated scheduling; investigate data o...
1,Each cycle,"Missingness, zero-position rate, and sudden me...",Stop scoring affected rows and investigate ins...
2,When later labels exist,"Grouped/time-aware precision@K, ROC AUC, avera...",Retrain only if validation and leakage audit a...
3,On major change,"Search, tracking, site, content-template, or c...",Treat prior estimates as stale; revalidate bef...
4,After actions,"Reviewer decisions and outcomes, with selectio...",Do not claim impact without an appropriate exp...


## 5. Exports for the paper

The queue CSV is written to `work/outputs/` and is intentionally not committed because it contains actionable pseudonymized identifiers. The summary metric receipt is committed. The figure uses only aggregated archetype counts and is saved to `work/figures/` for the public paper.

In [6]:
queue_export_columns = [
    'queue_rank', 'content_id', 'client_id', 'recommended_human_action', 'archetype',
    'reason_codes', 'review_score', 'priority_score', 'impressions_90d', 'clicks_90d',
    'avg_position', 'days_since_last_update', 'estimated_review_hours', 'value_proxy', 'review_rule'
]
queue_path = output_dir / 'w07_ranked_action_queue.csv'
queue[queue_export_columns].to_csv(queue_path, index=False)

archetype_counts = queue['archetype'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
archetype_counts.plot.barh(ax=ax, color='#2a6f97')
ax.set_title('Human review queue by action archetype')
ax.set_xlabel('Queued pages')
ax.set_ylabel('')
fig.tight_layout()
figure_path = figures_dir / 'w07_queue_by_archetype.png'
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.close(fig)

metrics = {
    'rows_scored': int(len(df)),
    'candidate_rows_with_500_plus_impressions': int(len(candidates)),
    'queue_rows': int(len(queue)),
    'pseudonymized_clients_represented': int(queue.client_id.nunique()),
    'per_client_cap': 3,
    'estimated_first_pass_review_hours': float(queue.estimated_review_hours.sum()),
    'validation_context': 'Week 6 client-grouped current-state proxy evaluation: ROC AUC 0.589, average precision 0.600, test base rate 0.517.',
    'queue_export': str(queue_path.relative_to(repo_root)),
    'figure_export': str(figure_path.relative_to(repo_root)),
    'not_for_automation': True
}
metrics_path = output_dir / 'w07_action_playbook_metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as handle:
    json.dump(metrics, handle, indent=2)

print(f'Wrote regenerateable queue: {queue_path.relative_to(repo_root)}')
print(f'Wrote reusable figure: {figure_path.relative_to(repo_root)}')
print(f'Wrote metrics receipt: {metrics_path.relative_to(repo_root)}')
display(pd.DataFrame([
    ('queue CSV', queue_path.relative_to(repo_root), 'not committed'),
    ('figure', figure_path.relative_to(repo_root), 'committed for paper reuse'),
    ('metrics JSON', metrics_path.relative_to(repo_root), 'committed as a receipt'),
], columns=['artifact', 'path', 'treatment']))

Wrote regenerateable queue: work\outputs\w07_ranked_action_queue.csv
Wrote reusable figure: work\figures\w07_queue_by_archetype.png
Wrote metrics receipt: work\outputs\w07_action_playbook_metrics.json


,artifact,path,treatment
0,queue CSV,work\outputs\w07_ranked_action_queue.csv,not committed
1,figure,work\figures\w07_queue_by_archetype.png,committed for paper reuse
2,metrics JSON,work\outputs\w07_action_playbook_metrics.json,committed as a receipt


## Self-check

- [x] I created a ranked, diverse, human-review queue with reason codes and archetype-to-action mapping.
- [x] I stated the decay/refresh insight without claiming that a refresh causes recovery.
- [x] I documented intended use, limits, cost/value thinking, review rules, and no-go cases.
- [x] I listed monitoring and retrain triggers that require fresh honest validation.
- [x] I export the actionable queue to `work/outputs/`, a public-safe aggregate figure to `work/figures/`, and a metric receipt to `work/outputs/`.
- [x] No client names, URLs, or raw queries are displayed.
- [x] I executed this notebook top to bottom and verified the exports.
- [x] I committed the notebook, public-safe figure, and metrics receipt to the repository.